# Logging & Error‑Proof Scraping (Quick Notes)

## Logging & Automation Basics

**Logging** is the practice of recording what a program is doing while it runs. It is better than using `print()` because logs can be saved, filtered, and reviewed later.

### Why Logging?

* Helps debug errors
* Tracks program flow
* Useful for automated scripts
* Keeps run history

### Common Log Levels

* **DEBUG** – Detailed developer info
* **INFO** – Normal operations
* **WARNING** – Unexpected but not fatal
* **ERROR** – Operation failed
* **CRITICAL** – Program cannot continue

### Simple Example

```python
import logging
logging.basicConfig(level=logging.INFO)
logging.info("Scraping started")
```
# Automation
**Automation** means running tasks automatically without manual input, such as scraping data, saving results, and running scripts on a schedule.
In scraping, automation includes:

* Automatically visiting pages
* Extracting data
* Saving results
* Running scripts on schedule

---

## Where Automation is Used

* Web scraping bots
* Data collection pipelines
* Scheduled reports
* Monitoring websites

---

## Key Automation Concepts (Should Know)

* Scripts should run without user input
* Handle failures automatically
* Save logs and outputs
* Can be scheduled (daily, weekly)

---

## Error‑Proof Scraping Design

**Error‑proof scraping** means writing scraping code that does not crash when problems occur.

### Common Scraping Issues

* Network errors
* Page not found (404)
* Website structure changes
* Missing or empty data

### Best Practices

* Use `try-except` to handle errors
* Never assume data always exists
* Use `.get()` for safe dictionary access
* Validate extracted data
* Log errors instead of stopping the script
* Continue scraping even if one item fails

### Example

```python
price = item.get("price", None)
if price is None:
    logging.warning("Price missing")
```

---

## Key Takeaway

* Logging makes automation **traceable and debuggable**
* Error‑proof design makes scraping **stable and reliable**
* Both are essential for **real‑world scraping projects**


**1. Add try–except blocks around all network requests to handle common errors such as ConnectionError, Timeout, and unexpected exceptions.**

In [9]:
import requests

url = "https://example.com"

try:
    response = requests.get(url, timeout=5)
    response.raise_for_status()   # raises error for 4xx/5xx
    print("Page fetched successfully")

except requests.exceptions.ConnectionError:
    print("Connection error: Unable to connect to the website")

except requests.exceptions.Timeout:
    print("Timeout error: The request took too long")

except requests.exceptions.RequestException as e:
    print("Request failed:", e)

except Exception as e:
    print("Unexpected error:", e)


Page fetched successfully


**2. Configure Python logging:
• Create a logger that writes logs to a file (scraper.log)
• Log INFO messages for normal flow (page fetched, items parsed)
• Log ERROR messages when exceptions occur**

In [18]:
import logging
import requests

# Logging Configuration

logging.basicConfig(
    filename="scraper.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s" 
    # %(asctime)s Date & time of log  --  %(levelname)s	Log level (INFO, ERROR) --  %(message)s	Your log message

)         

url = "https://example.com"

try:
    logging.info("Starting page fetch")

    response = requests.get(url, timeout=10)
    response.raise_for_status()

    logging.info("Page fetched successfully")

    # Example parsing step
    logging.info("Items parsed successfully")

except requests.exceptions.RequestException as e:
    logging.error(f"Request failed: {e}")
    

except Exception as e:
    logging.error(f"Unexpected error occurred: {e}")


**3.Implement a retry mechanism for failed requests:
• Retry a request up to N times (for example, 3 retries)
• Add a small delay between retries
• Stop retrying after max attempts and log the failure**

In [25]:
import requests
import logging
import time

# Logging configuration
logging.basicConfig(
    filename="scraper.log",
    level =logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s "
)
url = "https://example1.com"
MAX_RETRIES= 3
DELAY =2


for attempt in range(1,MAX_RETRIES+1):
    try:
        logging.info(f"Attempting {attempt} attempts.")
        response = requests.get(url,timeout=1)
        response.raise_for_status()
        logging.info("Page Fetched Successfully")
        break       # if success stops 
    
    except requests.exceptions.RequestException as e:
        logging.error(f"Request Failed on {attempt} attempt: {e} ")
    
        if attempt>=MAX_RETRIES:  # if 3=3 max attempt will print and the loops stops
            logging.info("Max attempt Reached: Request Failed")
        else:
            logging.info(f"Retrying after {DELAY} seconds...")
            time.sleep(DELAY)   # will be 2 seconds as we use 2 sec in above


**4. Modify fetch_page() to:
• Log the page number being scraped
• Log success when status code is 200
• Log warning or error when status code is not 200**

In [33]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import logging

# Logging configuration
logging.basicConfig(
    filename="scraper.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

def fetch_page(url):
    """Generator that yields HTML of each page until no next page exists, with logging."""
    page_number = 1

    while url:
        logging.info(f"Scraping page {page_number}: {url}")

        try:
            response = requests.get(url, timeout=10)

            if response.status_code == 200:
                logging.info(f"Page {page_number} fetched successfully (200)")
            else:
                logging.warning(f"Page {page_number} returned status code {response.status_code}")

            html = response.text
            yield html

        except requests.exceptions.RequestException as e:
            logging.error(f"Error fetching page {page_number}: {e}")
            break  # stop if request fails

        # Find next page
        soup = BeautifulSoup(html, "html.parser")
        next_link = soup.find("li", class_="next")
        if not next_link or not next_link.a:
            logging.info("No more pages to scrape. Stopping.")
            break

        url = urljoin(url, next_link.a["href"])
        page_number += 1
gen = fetch_page("http://books.toscrape.com/catalogue/page-1.html")

for i in gen:
    pass

**5. Ensure the scraper does not crash on a single failure:
• If one page fails, handle it gracefully
• Continue or stop based on your retry logic**

In [39]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import logging
import time

# Logging configuration
logging.basicConfig(
    filename="scraper.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

MAX_RETRIES = 3
DELAY = 2  # seconds

def fetch_page(url):
    """Generator that yields HTML of each page, with retry and graceful failure."""
    page_number = 1

    while url:
        logging.info(f"Scraping page {page_number}: {url}")

        success = False
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                logging.info(f"Attempt {attempt} to fetch page {page_number}")
                response = requests.get(url, timeout=10)
                response.raise_for_status()  # Raises HTTPError for bad status

                if response.status_code == 200:
                    logging.info(f"Page {page_number} fetched successfully (200)")
                    success = True
                    html = response.text
                    yield html
                    break  # stop retrying after success
                else:
                    logging.warning(f"Page {page_number} returned status code {response.status_code}")

            except requests.exceptions.RequestException as e:
                logging.error(f"Request failed on attempt {attempt}: {e}")
                if attempt < MAX_RETRIES:
                    logging.info(f"Retrying after {DELAY} seconds...")
                    time.sleep(DELAY)

        if not success:
            logging.error(f"Failed to fetch page {page_number} after {MAX_RETRIES} attempts. Skipping...")
            # Decide: either continue to next page or break
            # Here we continue scraping next pages
            html = ""  # yield empty HTML so generator continues
            yield html

        # Find next page
        try:
            soup = BeautifulSoup(html, "html.parser")
            next_link = soup.find("li", class_="next")
            if not next_link or not next_link.a:
                logging.info("No more pages to scrape. Stopping.")
                break
            url = urljoin(url, next_link.a["href"])
            page_number += 1
        except Exception as e:
            logging.error(f"Error parsing page {page_number}: {e}")
            break
gen = fetch_page("http://books.toscrape.com/catalogue/page-1.html")

for html in gen:
    pass 


**6. Add a final summary log at the end of execution:
• Total pages attempted
• Total pages successfully scraped
• Total failures**

In [30]:
import requests
import time
import logging
from bs4 import BeautifulSoup
from urllib.parse import urljoin

logging.basicConfig(
    filename = "raper.log",
    level = logging.INFO,
    format = "%(asctime)s - %(levelname)s - %(message)s "
)

MAX_RETRIES=3
DELAY= 2
def fetch_page(url):
    page_number =1
    total_pages=0
    failed_pages=0
    success_pages=0
    while url:
        logging.info(f"Scraping page {page_number}: {url}")
        total_pages+=1
        success = False
        for attempt in range(1,MAX_RETRIES+1):
            try:
                response = requests.get(url, timeout= 5)
                response.raise_for_status()

                if response.status_code == 200:
                    logging.info("Sucess(200)")
                    success_pages+=1
                    html = response.text
                    yield html
                    success =True
                    break
                else:
                    logging.warning(f"Waring on scraping pages {page_number}")
                    
            except requests.exceptions.RequestException as e:
                logging.error(f"Request failed : as it return status code {response.status_code}")
                if attempt < MAX_RETRIES:
                    logging.info(f"Delaying for 2 second attempt = {attempt}")
                    time.sleep(DELAY)
        if not success:
            failed_pages+=1
            logging.error(f"Something at pages {page_number}")
            html = ""
            yield html

        soup = BeautifulSoup(html, "html.parser")
        next_link = soup.find("li", class_="next")
        if not next_link or not next_link.a:
            logging.info("No more pages to scrape. Stopping.")
            break
        url = urljoin(url, next_link.a["href"])
        page_number += 1
    logging.info(f"Total_pages:{total_pages}")
    logging.info(f"failed_pages:{failed_pages}")
    logging.info(f"Success_pages:{success_pages}")
    
gen =fetch_page("http://books.toscrape.com/catalogue/page-1.html")
for i in gen: 
    pass
            

In [41]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import logging
import time

# Logging configuration
logging.basicConfig(
    filename="scraper.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

MAX_RETRIES = 3
DELAY = 2  # seconds

def fetch_page(url):
    page_number = 1
    total_pages = 0
    success_pages = 0
    failed_pages = 0

    while url:
        logging.info(f"Scraping page {page_number}: {url}")
        total_pages += 1
        success = False

        for attempt in range(1, MAX_RETRIES + 1):
            try:
                logging.info(f"Attempt {attempt} to fetch page {page_number}")
                response = requests.get(url, timeout=10)
                response.raise_for_status()

                logging.info(f"Page {page_number} fetched successfully (200)")
                success_pages += 1
                html = response.text
                yield html
                success = True
                break

            except requests.exceptions.RequestException as e:
                logging.error(f"Request failed on attempt {attempt}: {e}")
                if attempt < MAX_RETRIES:
                    logging.info(f"Retrying after {DELAY} seconds...")
                    time.sleep(DELAY)

        if not success:
            logging.error(f"Failed to fetch page {page_number} after {MAX_RETRIES} attempts")
            failed_pages += 1
            html = ""  # continue to next page
            yield html

        # Find next page
        try:
            soup = BeautifulSoup(html, "html.parser")
            next_link = soup.find("li", class_="next")
            if not next_link or not next_link.a:
                logging.info("No more pages to scrape. Stopping.")
                break
            url = urljoin(url, next_link.a["href"])
            page_number += 1
        except Exception as e:
            logging.error(f"Error parsing page {page_number}: {e}")
            break

    # Final summary log
    logging.info("---------- SCRAPING SUMMARY ----------")
    logging.info(f"Total pages attempted: {total_pages}")
    logging.info(f"Total pages successfully scraped: {success_pages}")
    logging.info(f"Total failures: {failed_pages}")

gen = fetch_page("http://books.toscrape.com/catalogue/page-1.html")
for html in gen:
    pass 


**7. Automation check
• Wrap the scraper execution inside a main() function
• Make sure the script can be scheduled or run automatically without manual intervention**

In [43]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import logging
import time

# ---------------- Logging Configuration ----------------
logging.basicConfig(
    filename="scraper.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

MAX_RETRIES = 3
DELAY = 2  # seconds

# ---------------- Scraper Function ----------------
def fetch_page(url):
    page_number = 51
    while url:
        logging.info(f"Scraping page {page_number}: {url}")
        success = False

        for attempt in range(1, MAX_RETRIES + 1):
            try:
                logging.info(f"Attempt {attempt} to fetch page {page_number}")
                response = requests.get(url, timeout=10)
                response.raise_for_status()
                logging.info(f"Page {page_number} fetched successfully (200)")
                html = response.text
                success = True
                yield html
                break
            except requests.exceptions.RequestException as e:
                logging.error(f"Request failed on attempt {attempt}: {e}")
                if attempt < MAX_RETRIES:
                    logging.info(f"Retrying after {DELAY} seconds...")
                    time.sleep(DELAY)

        if not success:
            logging.error(f"Failed to fetch page {page_number} after {MAX_RETRIES} attempts")
            yield ""  # continue to next page

        # Find next page
        try:
            soup = BeautifulSoup(html, "html.parser")
            next_link = soup.find("li", class_="next")
            if not next_link or not next_link.a:
                logging.info("No more pages to scrape. Stopping.")
                break
            url = urljoin(url, next_link.a["href"])
            page_number += 1
        except Exception as e:
            logging.error(f"Error parsing page {page_number}: {e}")
            break

# ---------------- Main Function ----------------
def main():
    start_url = "https://books.toscrape.com/catalogue/page-51.html"
    for i, html in enumerate(fetch_page(start_url), 1):
        logging.info(f"Processing page {i}...")
        # Here you can call your parsing function
        time.sleep(1)  # optional delay for polite scraping

    logging.info("Scraper run completed.")

# ---------------- Automatic Execution ----------------
if __name__ == "__main__":
    main()

# It checks if the script is being run directly
# If yes → execute main() (or any code inside this block)
# If no → do not run it automatically (e.g., when imported as a module)
